In [1]:
!pip install crewai crewai-tools groq pydantic qdrant-client sentence-transformers rank-bm25

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.3/89.3 kB 7.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 6.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.0 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of instructor to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 9.7 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of transformers to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 5.0 M

In [2]:
from google.colab import userdata
import json
import re
import numpy as np
from pydantic import BaseModel
from typing import List, Optional, Any

# Load all API keys
groq_api_key = userdata.get('GROQ_API_KEY')
qdrant_api_key = userdata.get('QDRANT_API_KEY')
qdrant_url = userdata.get('QDRANT_URL')

print("All API keys loaded successfully")

All API keys loaded successfully


In [3]:
from groq import Groq
from qdrant_client import QdrantClient
from qdrant_client.models import SparseVector, Prefetch, FusionQuery, Fusion
from sentence_transformers import SentenceTransformer, CrossEncoder

# ── SCHEMAS (from NB1) ─────────────────────────────────────
class Experience(BaseModel):
    role: str
    company: str
    duration: str
    description: str

class Education(BaseModel):
    degree: str
    institution: str
    year: Optional[Any] = None

class ParsedResume(BaseModel):
    name: str
    email: Optional[str] = None
    phone: Optional[str] = None
    skills: List[str]
    experience: List[Experience]
    education: List[Education]
    certifications: List[str] = []
    total_experience_years: Optional[float] = None

class ParsedJD(BaseModel):
    job_title: str
    company: Optional[str] = None
    required_skills: List[str]
    preferred_skills: List[str] = []
    minimum_experience_years: Optional[float] = None
    education_requirement: Optional[str] = None
    certifications_required: List[str] = []
    responsibilities: List[str] = []
    location: Optional[str] = None

print("All schemas loaded")

# ── LOAD MODELS ────────────────────────────────────────────
print("Loading BGE model...")
embedding_model = SentenceTransformer("BAAI/bge-large-en-v1.5")
print("Loading Cross-Encoder...")
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("Loading complete")

# ── CONNECT TO QDRANT ──────────────────────────────────────
qdrant_client = QdrantClient(url=qdrant_url, api_key=qdrant_api_key)
print("Qdrant connected")
print("Collections:", qdrant_client.get_collections())

All schemas loaded
Loading BGE model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Loading complete
Qdrant connected
Collections: collections=[CollectionDescription(name='resume_matcher')]


In [4]:
# ── VOCABULARY AND SPARSE VECTOR (from NB2) ────────────────
resume_corpus = [
    "Python Machine Learning Deep Learning NLP TensorFlow PyTorch SQL Docker Git FastAPI Scikit-learn Pandas NumPy BERT text classification deployment AWS EC2",
    "Java Spring Boot Microservices Kubernetes AWS Docker Jenkins CICD REST APIs backend software engineer",
    "Python NLP Transformers BERT LangChain RAG LLM HuggingFace Vector Databases Qdrant embeddings semantic search",
    "Python Data Engineering Spark Hadoop Airflow ETL Pipeline SQL PostgreSQL AWS S3 data warehouse",
    "Python TensorFlow Computer Vision OpenCV YOLO Object Detection Image Classification CNN ResNet",
]

def text_to_tokens(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    return [t for t in text.split() if len(t) > 1]

all_tokens = []
for doc in resume_corpus:
    all_tokens.extend(text_to_tokens(doc))
vocabulary = {token: idx for idx, token in enumerate(set(all_tokens))}

def build_sparse_vector(text):
    tokens = text_to_tokens(text)
    tf = {}
    for token in tokens:
        if token in vocabulary:
            idx = vocabulary[token]
            tf[idx] = tf.get(idx, 0) + 1
    if not tf:
        return SparseVector(indices=[0], values=[0.0])
    max_freq = max(tf.values())
    return SparseVector(
        indices=list(tf.keys()),
        values=[round(float(v)/max_freq, 4) for v in tf.values()]
    )

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# ── PARSER FUNCTIONS (from NB1) ────────────────────────────
def parse_resume(resume_text):
    client = Groq(api_key=groq_api_key)
    prompt = f"""
    You are an expert resume parser for a production ATS system.
    Extract all information from the resume below.
    Return ONLY a valid JSON object, no markdown, no explanation.
    JSON fields: name, email, phone, skills (list),
    experience (list of role/company/duration/description),
    education (list of degree/institution/year),
    certifications (list), total_experience_years (float)

    Resume:
    {resume_text}
    """
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "You are a resume parser. Return valid JSON only."},
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )
    raw = response.choices[0].message.content.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    return ParsedResume(**json.loads(raw))

def parse_jd(jd_text):
    client = Groq(api_key=groq_api_key)
    prompt = f"""
    You are an expert Job Description parser for a production ATS system.
    Extract all information from the JD below.
    Return ONLY a valid JSON object, no markdown, no explanation.
    JSON fields: job_title, company, required_skills (list),
    preferred_skills (list), minimum_experience_years (float),
    education_requirement, certifications_required (list),
    responsibilities (list), location

    Job Description:
    {jd_text}
    """
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "You are a JD parser. Return valid JSON only."},
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )
    raw = response.choices[0].message.content.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    return ParsedJD(**json.loads(raw))

# ── ATS SCORING FUNCTIONS (from NB2) ──────────────────────
def calculate_skills_match(candidate_skills, required_skills, preferred_skills):
    candidate_lower = [s.lower().strip() for s in candidate_skills]
    required_matched, required_missing = [], []
    for skill in required_skills:
        skill_lower = skill.lower().strip()
        matched = any(skill_lower in c or c in skill_lower for c in candidate_lower)
        if matched:
            required_matched.append(skill)
        else:
            required_missing.append(skill)
    preferred_matched, preferred_missing = [], []
    for skill in preferred_skills:
        skill_lower = skill.lower().strip()
        matched = any(skill_lower in c or c in skill_lower for c in candidate_lower)
        if matched:
            preferred_matched.append(skill)
        else:
            preferred_missing.append(skill)
    req_score = len(required_matched) / len(required_skills) * 100 if required_skills else 100
    pref_score = len(preferred_matched) / len(preferred_skills) * 100 if preferred_skills else 100
    return req_score, pref_score, required_matched, required_missing, preferred_matched, preferred_missing

def calculate_experience_score(candidate_exp, min_required):
    if not min_required or min_required == 0:
        return 100
    if not candidate_exp:
        return 0
    if candidate_exp >= min_required:
        return 100
    return round((candidate_exp / min_required) * 100, 2)

def hybrid_search_and_score(jd, top_k=5):
    query_text = f"{jd.job_title} {' '.join(jd.required_skills)} {' '.join(jd.preferred_skills)}"
    dense_vector = embedding_model.encode(query_text, normalize_embeddings=True).tolist()
    sparse_vector = build_sparse_vector(query_text)
    results = qdrant_client.query_points(
        collection_name="resume_matcher",
        prefetch=[
            Prefetch(query=dense_vector, using="dense", limit=top_k),
            Prefetch(query=sparse_vector, using="sparse", limit=top_k)
        ],
        query=FusionQuery(fusion=Fusion.RRF),
        limit=top_k,
        with_payload=True
    )
    candidates = results.points
    pairs = [[query_text, c.payload["text"]] for c in candidates]
    raw_scores = cross_encoder.predict(pairs)
    semantic_scores = [round(float(sigmoid(s)) * 100, 2) for s in raw_scores]
    ranked = []
    for candidate, semantic_score in zip(candidates, semantic_scores):
        req_score, pref_score, req_matched, req_missing, pref_matched, pref_missing = calculate_skills_match(
            candidate.payload["skills"], jd.required_skills, jd.preferred_skills
        )
        exp_score = calculate_experience_score(
            candidate.payload.get("exp_years", 0),
            jd.minimum_experience_years
        )
        if req_score < 40:
            final_ats = round(req_score * 0.40, 2)
            status = "AUTO REJECTED"
        else:
            final_ats = round(
                (req_score * 0.40) + (semantic_score * 0.30) +
                (exp_score * 0.15) + (pref_score * 0.10) + 100 * 0.05,
                2
            )
            status = "SHORTLISTED" if final_ats >= 75 else "REVIEW" if final_ats >= 50 else "REJECTED"
        ranked.append({
            "name": candidate.payload["name"],
            "email": candidate.payload.get("email"),
            "skills": candidate.payload["skills"],
            "exp_years": candidate.payload.get("exp_years", 0),
            "education": candidate.payload.get("education"),
            "certifications": candidate.payload.get("certifications", []),
            "final_ats_score": final_ats,
            "status": status,
            "required_skills_score": round(req_score, 2),
            "semantic_score": semantic_score,
            "experience_score": exp_score,
            "preferred_skills_score": round(pref_score, 2),
            "required_matched": req_matched,
            "required_missing": req_missing,
            "preferred_matched": pref_matched,
            "preferred_missing": pref_missing
        })
    return sorted(ranked, key=lambda x: x["final_ats_score"], reverse=True)

print("All helper functions loaded successfully")
print("Parser functions ready")
print("ATS scoring functions ready")
print("Hybrid search ready")

All helper functions loaded successfully
Parser functions ready
ATS scoring functions ready
Hybrid search ready


In [5]:
from crewai import Agent, Task, Crew
from crewai.flow.flow import Flow, listen, start
from pydantic import BaseModel
from typing import Optional, List, Any

# ── FLOW STATE ─────────────────────────────────────────────
# This carries data between all 3 agents automatically
class ResumeMatcherState(BaseModel):
    # Inputs
    raw_resume: str = ""
    raw_jd: str = ""

    # Agent 1 outputs
    parsed_resume: Optional[Any] = None
    parsed_jd: Optional[Any] = None

    # Agent 2 outputs
    ats_results: Optional[List[Any]] = None
    top_candidate: Optional[Any] = None

    # Agent 3 outputs
    feedback_report: str = ""
    upskilling_roadmap: str = ""

print("Flow state defined")

# ── AGENT 1: PARSING AGENT ─────────────────────────────────
def agent1_parse(state: ResumeMatcherState) -> ResumeMatcherState:
    print("\nAGENT 1: Parsing Resume and JD...")

    parsed_resume = parse_resume(state.raw_resume)
    parsed_jd = parse_jd(state.raw_jd)

    state.parsed_resume = parsed_resume
    state.parsed_jd = parsed_jd

    print(f"  Resume parsed: {parsed_resume.name}")
    print(f"  Skills found: {len(parsed_resume.skills)}")
    print(f"  JD parsed: {parsed_jd.job_title} at {parsed_jd.company}")
    print(f"  Required skills: {len(parsed_jd.required_skills)}")
    print("AGENT 1: Done")

    return state

# ── AGENT 2: MATCHING AGENT ────────────────────────────────
def agent2_match(state: ResumeMatcherState) -> ResumeMatcherState:
    print("\nAGENT 2: Running Hybrid Search and ATS Scoring...")

    ats_results = hybrid_search_and_score(state.parsed_jd, top_k=5)

    state.ats_results = ats_results
    state.top_candidate = ats_results[0] if ats_results else None

    print(f"  Candidates scored: {len(ats_results)}")
    for r in ats_results:
        print(f"  {r['name']}: {r['final_ats_score']}/100 [{r['status']}]")
    print("AGENT 2: Done")

    return state

# ── AGENT 3: FEEDBACK AGENT ────────────────────────────────
def agent3_feedback(state: ResumeMatcherState) -> ResumeMatcherState:
    print("\nAGENT 3: Generating Qualitative Feedback and Upskilling Roadmap...")

    client = Groq(api_key=groq_api_key)

    # Find the submitted candidate in results
    candidate_name = state.parsed_resume.name
    candidate_result = next(
        (r for r in state.ats_results if candidate_name.lower() in r["name"].lower()),
        state.top_candidate
    )

    feedback_prompt = f"""
    You are a senior HR expert and career coach at a top tech company.
    A candidate has applied for a job and received an ATS score.

    Your task is to write TWO things:
    1. A detailed REVIEW explaining exactly why they received this score
    2. A personalized 90-DAY UPSKILLING ROADMAP to improve their chances

    CANDIDATE PROFILE:
    - Name: {state.parsed_resume.name}
    - Skills: {state.parsed_resume.skills}
    - Experience: {state.parsed_resume.total_experience_years} years
    - Education: {state.parsed_resume.education[0].degree} from {state.parsed_resume.education[0].institution}
    - Certifications: {state.parsed_resume.certifications}

    JOB DETAILS:
    - Role: {state.parsed_jd.job_title} at {state.parsed_jd.company}
    - Required Skills: {state.parsed_jd.required_skills}
    - Preferred Skills: {state.parsed_jd.preferred_skills}
    - Min Experience: {state.parsed_jd.minimum_experience_years} years

    ATS SCORING RESULTS:
    - Final ATS Score: {candidate_result['final_ats_score']}/100
    - Status: {candidate_result['status']}
    - Required Skills Score: {candidate_result['required_skills_score']}%
    - Skills Matched: {candidate_result['required_matched']}
    - Skills Missing: {candidate_result['required_missing']}
    - Preferred Skills Matched: {candidate_result['preferred_matched']}
    - Experience Score: {candidate_result['experience_score']}%

    Write a professional, honest, and encouraging response with:

    ## SCORE REVIEW
    Explain in detail why the candidate received this score.
    Be specific about what is strong and what is weak.

    ## GAP ANALYSIS
    List exactly what is missing and why it matters for this role.

    ## 90-DAY UPSKILLING ROADMAP
    Month 1: [Specific actions with resources]
    Month 2: [Specific actions with resources]
    Month 3: [Specific actions with resources]

    ## FINAL RECOMMENDATION
    What should the candidate do right now?
    """

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "You are a senior HR expert and career coach. Give honest, detailed, actionable feedback."},
            {"role": "user", "content": feedback_prompt}
        ],
        temperature=0.3
    )

    feedback = response.choices[0].message.content
    state.feedback_report = feedback

    print("AGENT 3: Feedback generated successfully")
    return state

print("All 3 agents defined successfully")

Flow state defined
All 3 agents defined successfully


In [10]:
# ── CREWAI FLOW ────────────────────────────────────────────
class ResumeMatcherFlow(Flow[ResumeMatcherState]):

    @start()
    def parsing_step(self):
        print("FLOW STARTED")
        print("="*60)
        updated_state = agent1_parse(self.state)
        return updated_state

    @listen(parsing_step)
    def matching_step(self, state):
        updated_state = agent2_match(state)
        return updated_state

    @listen(matching_step)
    def feedback_step(self, state):
        updated_state = agent3_feedback(state)
        return updated_state

print("CrewAI Flow defined successfully")

# ── RUN THE FULL PIPELINE ──────────────────────────────────
raw_resume = """
Charan Kumar
Email: charan@gmail.com
Phone: +91-9876543210

SKILLS
Python, Machine Learning, Deep Learning, NLP, TensorFlow, PyTorch,
SQL, Docker, Git, FastAPI, Scikit-learn, Pandas, NumPy

EXPERIENCE
Machine Learning Intern - TechStartup Pvt Ltd
June 2024 - December 2024
Built text classification models using BERT for customer feedback analysis.
Deployed ML models using FastAPI and Docker on AWS EC2.

Data Science Intern - Analytics Corp
January 2024 - May 2024
Performed exploratory data analysis on sales datasets using Pandas.
Created visualization dashboards using Matplotlib and Seaborn.

EDUCATION
Bachelor of Technology in Computer Science
IIIT Kota, 2025

CERTIFICATIONS
Deep Learning Specialization - Coursera (Andrew Ng)
AWS Cloud Practitioner
"""

raw_jd = """
Company: Google India
Role: Machine Learning Engineer

Required Skills:
Python, Machine Learning, Deep Learning, NLP, Docker, Git

Preferred Skills:
LangChain, HuggingFace, RAG, MLflow, Kubernetes

Minimum Experience: 1 year

Education: BTech or MTech in Computer Science

Responsibilities:
- Build and deploy production ML models
- Design and implement NLP pipelines
- Monitor model performance in production

Location: Bangalore, India
"""

# Initialize flow with inputs
initial_state = ResumeMatcherState(
    raw_resume=raw_resume,
    raw_jd=raw_jd
)

flow = ResumeMatcherFlow()
result = await flow.kickoff_async(inputs={
    "raw_resume": raw_resume,
    "raw_jd": raw_jd
})

# Show final results
print("\n" + "="*60)
print("FULL PIPELINE RESULTS")
print("="*60)
print("\nALL CANDIDATES ATS SCORES:")
print("-"*60)
for rank, c in enumerate(flow.state.ats_results, 1):
    print(f"Rank {rank}: {c['name']} | {c['final_ats_score']}/100 | {c['status']}")

print("\n" + "="*60)
print("FEEDBACK REPORT FOR:", flow.state.parsed_resume.name)
print("="*60)
print(flow.state.feedback_report)

CrewAI Flow defined successfully


Flow started with ID: 69568ab6-3ec2-4c58-aa91-9d2f1e011728

╭─────────────────────────────────────────────── 🌊 Flow Execution ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Starting Flow Execution                                                                                        │
│  Name:                                                                                                          │
│  ResumeMatcherFlow                                                                                              │
│  ID:                                                                                                            │
│  69568ab6-3ec2-4c58-aa91-9d2f1e011728                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

FLOW STARTED

AGENT 1: Parsing Resume and JD...


╭──────────────────────────────────────────────── 🌊 Flow Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Started                                                                                                   │
│  Name: ResumeMatcherFlow                                                                                        │
│  ID: 69568ab6-3ec2-4c58-aa91-9d2f1e011728                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭────────────────────────────────────────────── ✅ Flow Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  ResumeMatcherFlow                                                                                              │
│  ID:                                                                                                            │
│  69568ab6-3ec2-4c58-aa91-9d2f1e011728                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── Execution Traces ────────────────────────────────────────────────╮
│                                                                                                                 │
│  🔍 Detailed execution traces are available!                                                                    │
│                                                                                                                 │
│  View insights including:                                                                                       │
│    • Agent decision-making process                                                                              │
│    • Task execution flow and timing                                                                             │
│    • Tool usage details                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Would you like to view your execution traces? [y/N] (20s timeout): 

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


FULL PIPELINE RESULTS

ALL CANDIDATES ATS SCORES:
------------------------------------------------------------
Rank 1: Rahul Verma | 82.62/100 | SHORTLISTED
Rank 2: Charan Kumar | 81.42/100 | SHORTLISTED
Rank 3: Arjun Mehta | 13.33/100 | AUTO REJECTED
Rank 4: Priya Sharma | 6.67/100 | AUTO REJECTED
Rank 5: Sneha Patel | 6.67/100 | AUTO REJECTED

FEEDBACK REPORT FOR: Charan Kumar
## SCORE REVIEW
Charan Kumar received an ATS score of 81.42/100, which is a commendable score, especially considering the highly competitive nature of the Machine Learning Engineer role at Google India. The breakdown of the score reveals that Charan excels in the required skills department, with a perfect score of 100.0%. This is a testament to his strong foundation in Python, Machine Learning, Deep Learning, NLP, Docker, and Git, which are all essential skills for the role. The skills matched section confirms that he possesses all the necessary skills, which is a significant strength.

However, the experience

In [11]:
# Print full feedback report
print("ALL CANDIDATES ATS SCORES:")
print("-"*60)
for rank, c in enumerate(flow.state.ats_results, 1):
    print(f"Rank {rank}: {c['name']} | {c['final_ats_score']}/100 | {c['status']}")

print("\n" + "="*60)
print("FEEDBACK REPORT FOR:", flow.state.parsed_resume.name)
print("="*60)
print(flow.state.feedback_report)

ALL CANDIDATES ATS SCORES:
------------------------------------------------------------
Rank 1: Rahul Verma | 82.62/100 | SHORTLISTED
Rank 2: Charan Kumar | 81.42/100 | SHORTLISTED
Rank 3: Arjun Mehta | 13.33/100 | AUTO REJECTED
Rank 4: Priya Sharma | 6.67/100 | AUTO REJECTED
Rank 5: Sneha Patel | 6.67/100 | AUTO REJECTED

FEEDBACK REPORT FOR: Charan Kumar
## SCORE REVIEW
Charan Kumar received an ATS score of 81.42/100, which is a commendable score, especially considering the highly competitive nature of the Machine Learning Engineer role at Google India. The breakdown of the score reveals that Charan excels in the required skills department, with a perfect score of 100.0%. This is a testament to his strong foundation in Python, Machine Learning, Deep Learning, NLP, Docker, and Git, which are all essential skills for the role. The skills matched section confirms that he possesses all the necessary skills, which is a significant strength.

However, the experience score is 50.0%, indicat